# Text embeddings of the joint graph

Embeds `title + "\n\n" + abstract` of every paper in `data/processed/joint/nodes.csv` with
`voyage-4-large` and writes `data/processed/joint/node_embeddings.npy`, where row `i` is the
embedding of `node_id == i`. Progress is saved after every batch, so a rerun resumes where it
stopped. Requires `VOYAGE_API_KEY` in `.env`.

In [ ]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import voyageai
from dotenv import load_dotenv

MODEL      = "voyage-4-large"
INPUT_TYPE = "document"
BATCH_SIZE = 128          # Voyage max texts per request
MAX_RETRIES = 6

ROOT      = Path.cwd().parent
OUT_PATH  = ROOT / "data" / "processed" / "joint" / "node_embeddings.npy"
PROG_PATH = ROOT / "data" / "processed" / "joint" / "node_embeddings.progress.json"

load_dotenv(ROOT / ".env")
client = voyageai.Client(api_key=os.environ["VOYAGE_API_KEY"])
print("Voyage client ready")

In [ ]:
nodes = pd.read_csv(ROOT / "data" / "processed" / "joint" / "nodes.csv", usecols=["node_id", "title", "abstract"])

# Row index must equal node_id, otherwise the saved array can't be indexed by node_id.
assert (nodes["node_id"].to_numpy() == np.arange(len(nodes))).all(), "node_id is not 0..N-1 in row order"

texts = (nodes["title"].fillna("") + "\n\n" + nodes["abstract"].fillna("")).tolist()
texts = [t if t.strip() else " " for t in texts]   # Voyage rejects empty strings
N = len(texts)
print(f"{N:,} nodes loaded")

In [ ]:
def embed_batch(batch):
    """Embed one batch, retrying with exponential backoff. Raises if it never succeeds."""
    for attempt in range(MAX_RETRIES):
        try:
            return client.embed(batch, model=MODEL, input_type=INPUT_TYPE).embeddings
        except Exception as e:
            wait = 2 ** attempt
            print(f"    {type(e).__name__}: {e} — retrying in {wait}s ({attempt + 1}/{MAX_RETRIES})", flush=True)
            time.sleep(wait)
    raise RuntimeError(f"batch failed after {MAX_RETRIES} attempts")

In [ ]:
# Resume from a previous run if there is one.
if PROG_PATH.exists():
    prog = json.loads(PROG_PATH.read_text())
    assert prog["model"] == MODEL, f"existing run used {prog['model']}, not {MODEL}"
    rows_done = prog["rows_done"]
    emb = np.lib.format.open_memmap(OUT_PATH, mode="r+")
    print(f"Resuming: {rows_done:,}/{N:,} rows already embedded ({rows_done / N:.1%})")
else:
    rows_done, emb = 0, None
    print(f"Starting fresh: {N:,} rows to embed")

session_start = rows_done
t0 = time.time()
for start in range(rows_done, N, BATCH_SIZE):
    batch = texts[start:start + BATCH_SIZE]
    vectors = np.asarray(embed_batch(batch), dtype=np.float32)

    if emb is None:   # dimension is only known after the first response
        emb = np.lib.format.open_memmap(
            OUT_PATH, mode="w+", dtype=np.float32, shape=(N, vectors.shape[1]))
        print(f"Created {OUT_PATH.name} with shape ({N:,}, {vectors.shape[1]})")

    emb[start:start + len(batch)] = vectors
    emb.flush()

    rows_done = start + len(batch)
    PROG_PATH.write_text(json.dumps({"rows_done": rows_done, "model": MODEL}))

    batch_no = start // BATCH_SIZE + 1
    if batch_no % 10 == 0 or rows_done == N:
        elapsed = time.time() - t0
        rate = (rows_done - session_start) / max(elapsed, 1e-9)
        eta = (N - rows_done) / max(rate, 1e-9)
        print(f"  {rows_done:,}/{N:,} ({rows_done / N:.1%})  "
              f"elapsed {elapsed / 60:.1f} min  ETA {eta / 60:.1f} min", flush=True)

print(f"\nDone — {rows_done:,} rows embedded in {(time.time() - t0) / 60:.1f} min")

In [ ]:
saved = np.load(OUT_PATH, mmap_mode="r")
print(f"{OUT_PATH}")
print(f"  shape : {saved.shape}")
print(f"  dtype : {saved.dtype}")
print(f"  size  : {OUT_PATH.stat().st_size / 1e9:.2f} GB")
print(f"  all-zero rows (never written): {int((np.abs(saved).sum(axis=1) == 0).sum()):,}")